# 🏥 Hospital Readmission Prediction 2.0

A learning/portfolio project based on the Diabetes 130-US Hospitals dataset. This version adds two features: **readmission probability/risk level** and a **Streamlit prediction app**.

> Educational project only. This model is not clinically validated and must not be used for medical decisions.

## 1. Install / import libraries

Run this cell in Google Colab.

In [1]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

import os, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 2. Download the dataset

The notebook downloads the public UCI Diabetes 130-US Hospitals dataset automatically, so you do not have to upload the large CSV manually.

In [2]:
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
zip_path = os.path.join(DATA_DIR, 'diabetes.zip')
url = 'https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip'

if not os.path.exists(os.path.join(DATA_DIR, 'diabetic_data.csv')):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)

csv_path = os.path.join(DATA_DIR, 'diabetic_data.csv')
df = pd.read_csv(csv_path)
print('Shape:', df.shape)
df.head()

Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 3. Understand the target

The original `readmitted` column has `<30`, `>30`, and `NO`. For this project we define the positive class as **readmitted within 30 days** (`<30`).

In [3]:
print(df['readmitted'].value_counts(dropna=False))

df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
print('\nBinary target distribution:')
print(df['readmitted_30'].value_counts())
print(df['readmitted_30'].value_counts(normalize=True).round(3))

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Binary target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64
readmitted_30
0    0.888
1    0.112
Name: proportion, dtype: float64


## 4. Select useful features

To keep the first version understandable and practical, we use a smaller set of demographic, utilization, and hospital-stay features. We also remove identifiers to reduce the risk of learning meaningless IDs.


In [4]:
features = [
    'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
    'admission_source_id', 'time_in_hospital', 'num_lab_procedures',
    'num_procedures', 'num_medications', 'number_outpatient',
    'number_emergency', 'number_inpatient', 'number_diagnoses',
    'change', 'diabetesMed'
]

model_df = df[features + ['readmitted_30']].copy()

# Convert age brackets such as [70-80) to the midpoint 75.
age_bounds = model_df['age'].str.extract(r'\[(\d+)-(\d+)\)')
model_df['age'] = age_bounds.astype(float).mean(axis=1)

X = model_df[features]
y = model_df['readmitted_30']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print('Train:', X_train.shape)
print('Test :', X_test.shape)

Train: (81412, 16)
Test : (20354, 16)


In [5]:
display(model_df['age'].value_counts().sort_index())

,count
age,
0.0,161
10.0,691
20.0,1657
30.0,3775
40.0,9685
50.0,17256
60.0,22483
70.0,26068
80.0,17197


## 5. Preprocessing pipeline

Categorical columns are imputed and one-hot encoded. Numerical columns are imputed and scaled.

In [ ]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

## 6. Train two models

We compare Logistic Regression and Random Forest. The model with the better ROC-AUC on the test set is selected for the final predictor.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, max_depth=12, min_samples_leaf=3,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )
}

results = []
fitted = {}

for name, estimator in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob)
    }
    results.append(metrics)
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = fitted[best_name]
print('Selected model:', best_name)

best_pred = best_model.predict(X_test)
print(classification_report(y_test, best_pred, target_names=['Not readmitted <30 days', 'Readmitted <30 days']))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_name}')
plt.show()

## 8. FEATURE 1 — Patient Risk Score

The application converts the model probability into a simple risk level:

- 0–30% → Low
- 30–60% → Medium
- 60–100% → High

These thresholds are **project-defined**, not clinical standards.

In [ ]:
def risk_level(probability):
    if probability < 0.30:
        return 'LOW'
    elif probability < 0.60:
        return 'MEDIUM'
    return 'HIGH'

sample_prob = best_model.predict_proba(X_test.iloc[[0]])[0, 1]
print(f'Readmission probability: {sample_prob:.1%}')
print('Risk level:', risk_level(sample_prob))

## 9. Test with a patient input

Edit the values below and run the cell.

In [ ]:
patient = pd.DataFrame([{
    'race': 'Caucasian',
    'gender': 'Female',
    'age': 75,
    'admission_type_id': 1,
    'discharge_disposition_id': 1,
    'admission_source_id': 7,
    'time_in_hospital': 6,
    'num_lab_procedures': 45,
    'num_procedures': 1,
    'num_medications': 12,
    'number_outpatient': 1,
    'number_emergency': 0,
    'number_inpatient': 2,
    'number_diagnoses': 7,
    'change': 'Ch',
    'diabetesMed': 'Yes'
}])

prob = best_model.predict_proba(patient)[0, 1]
prediction = int(prob >= 0.50)

print('Prediction:', 'Readmitted within 30 days' if prediction else 'Not readmitted within 30 days')
print(f'Readmission probability: {prob:.1%}')
print('Risk level:', risk_level(prob))

## 10. Save the trained model

This creates a model file that can be loaded by the Streamlit app.

In [ ]:
os.makedirs('/content/model', exist_ok=True)
joblib.dump(best_model, '/content/model/readmission_model.joblib')
joblib.dump(features, '/content/model/features.joblib')
print('Saved: /content/model/readmission_model.joblib')
print('Saved: /content/model/features.joblib')

## 11. Optional: create the Streamlit app file

The generated `app.py` is intended to be run locally or deployed with Streamlit. It is not necessary to run Streamlit inside this notebook.

In [ ]:
app_code = '''
import joblib
import pandas as pd
import streamlit as st

MODEL_PATH = "model/readmission_model.joblib"
model = joblib.load(MODEL_PATH)

FEATURES = [
    "race", "gender", "age", "admission_type_id",
    "discharge_disposition_id", "admission_source_id",
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses", "change", "diabetesMed"
]

def risk_level(probability):
    if probability < 0.30:
        return "LOW"
    if probability < 0.60:
        return "MEDIUM"
    return "HIGH"

st.set_page_config(page_title="Hospital Readmission Predictor", page_icon="🏥")
st.title("🏥 Hospital Readmission Predictor")
st.caption("Educational portfolio project — not for clinical decision-making.")

with st.form("patient_form"):
    race = st.selectbox(
        "Race",
        ["Caucasian", "AfricanAmerican", "Asian", "Hispanic", "Other"]
    )
    gender = st.selectbox("Gender", ["Female", "Male"])
    age = st.number_input("Age", min_value=0.0, max_value=95.0, value=65.0)
    admission_type_id = st.number_input("Admission Type ID", min_value=1, max_value=8, value=1)
    discharge_disposition_id = st.number_input(
        "Discharge Disposition ID", min_value=1, max_value=30, value=1
    )
    admission_source_id = st.number_input(
        "Admission Source ID", min_value=1, max_value=25, value=7
    )
    time_in_hospital = st.number_input(
        "Time in Hospital", min_value=1, max_value=14, value=5
    )
    num_lab_procedures = st.number_input(
        "Number of Lab Procedures", min_value=0, max_value=200, value=40
    )
    num_procedures = st.number_input(
        "Number of Procedures", min_value=0, max_value=10, value=1
    )
    num_medications = st.number_input(
        "Number of Medications", min_value=0, max_value=100, value=10
    )
    number_outpatient = st.number_input(
        "Outpatient Visits", min_value=0, max_value=100, value=0
    )
    number_emergency = st.number_input(
        "Emergency Visits", min_value=0, max_value=100, value=0
    )
    number_inpatient = st.number_input(
        "Inpatient Visits", min_value=0, max_value=100, value=0
    )
    number_diagnoses = st.number_input(
        "Number of Diagnoses", min_value=1, max_value=20, value=5
    )
    change = st.selectbox("Medication Change", ["No", "Ch"])
    diabetes_med = st.selectbox("Diabetes Medication", ["Yes", "No"])
    submitted = st.form_submit_button("Predict")

if submitted:
    patient = pd.DataFrame([{
        "race": race,
        "gender": gender,
        "age": age,
        "admission_type_id": admission_type_id,
        "discharge_disposition_id": discharge_disposition_id,
        "admission_source_id": admission_source_id,
        "time_in_hospital": time_in_hospital,
        "num_lab_procedures": num_lab_procedures,
        "num_procedures": num_procedures,
        "num_medications": num_medications,
        "number_outpatient": number_outpatient,
        "number_emergency": number_emergency,
        "number_inpatient": number_inpatient,
        "number_diagnoses": number_diagnoses,
        "change": change,
        "diabetesMed": diabetes_med
    }])[FEATURES]

    probability = model.predict_proba(patient)[0, 1]
    level = risk_level(probability)

    st.subheader("Prediction")
    st.metric("Readmission probability", f"{probability:.1%}")
    st.write(f"### Risk level: {level}")

    if probability >= 0.50:
        st.warning("The model predicts higher probability of readmission within 30 days.")
    else:
        st.success("The model predicts lower probability of readmission within 30 days.")
'''

with open('/content/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('Created /content/app.py')


## 12. Run the Streamlit app (locally)

Once the `app.py` file is created (after running the cell above), you can run it using the Streamlit CLI.

**To run it locally on your machine:**

1.  **Save the `app.py` file:** Download the `app.py` file from the `/content/` directory in your Colab environment to your local machine.
2.  **Install Streamlit:** If you haven't already, install Streamlit in your local Python environment:
    ```bash
    pip install streamlit
    ```
3.  **Navigate to the directory:** Open your terminal or command prompt, and navigate to the directory where you saved `app.py`.
4.  **Run the app:** Execute the following command:
    ```bash
    streamlit run app.py
    ```

This will open the Streamlit application in your web browser.